In [101]:
from itertools import pairwise
from datetime import date, timedelta
import pandas as pd

from tdf_pool.race import Race
from tdf_pool.cycling_calendar import get_calendar
from tdf_pool.score import score_race, get_score_template

def calendar_to_race_list(calendar: pd.DataFrame) -> list[Race]:
    races = []
    for _, row in calendar.iterrows():
        races.append(Race(row['Name'], row['Start'], row['Type'], row['PartialURL']))

    return races


def calendar_between(date_start: date, date_end: date, **filters) -> pd.DataFrame:
    # Figure out which years to get the calendar for
    years = list(range(date_start.year, date_end.year+1))

    # Get the calenders of these years
    calendars = [get_calendar(year) for year in years]
    if calendars == []:
        return pd.DataFrame(data=[], columns=["Start", "End", "Name", "PartialURL","Type"])
    calendar = pd.concat(calendars, axis=0)

    # Filter the exact dates
    calendar = calendar.loc[calendar['Start'].between(date_start, date_end)]

    # Apply all other filters
    if filters:
        # Make sure all filter items are lists
        for key, items in filters.items():
            if not isinstance(items, list):
                filters[key] = [items]

        # Select rows for which all filters are True
        calendar = calendar.loc[calendar[filters.keys()].isin(filters).all(axis=1)]

    # Cleanup the index
    calendar = calendar.reset_index(drop=True)
    return calendar

def combine_scores(scores: pd.DataFrame, score_mapping:dict[str,list[str]])-> pd.DataFrame:
    for key, columns in score_mapping.items():
        scores[key] = scores.loc[:,columns].sum(axis=1)
    
    cols = ['Rider', 'Team'] + list(score_mapping.keys())
    return scores[cols]

In [76]:

then, now = date(2015,1,1), date(2024, 1, 1)
tdfs_calendar = calendar_between(then, now, Name='Tour de France')
tdfs = calendar_to_race_list(tdfs_calendar)

In [86]:
intervals = [(tdfs[0].date - timedelta(30*(i)+1),  tdfs[0].date - timedelta(30*(i-1)+1)) for i in range(24)]

In [87]:
interval = intervals[0]
interval

(datetime.date(2015, 7, 3), datetime.date(2015, 8, 2))

In [88]:
interval_calendar = calendar_between(intervals[1][0], intervals[1][1])
interval_calendar

,Start,End,Name,PartialURL,Type
0,2015-06-07,2015-06-14,Critérium du Dauphiné,race/dauphine/2015/gc,2.UWT
1,2015-06-13,2015-06-21,Tour de Suisse,race/tour-de-suisse/2015/gc,2.UWT


In [90]:
interval_race_list = calendar_to_race_list(interval_calendar)
interval_race_list

[<Race: Critérium du Dauphiné, Date: 2015-06-07, number of stages: 8>,
 <Race: Tour de Suisse, Date: 2015-06-13, number of stages: 9>]

In [95]:
score_template = get_score_template()
interval_scores = [score_race(race, score_template) for race in interval_race_list]
interval_score = pd.concat(interval_scores).groupby(by=['Rider', 'Team']).sum().reset_index()

,Rider,Team,Stage,RedLantern,GC,KOM,Sprint,Youth,IntermediateKOM,Total,IntermediateSprint
62,DUMOULIN Tom,Team Giant - Alpecin,134.0,0.0,109.0,0.0,80.0,0.0,4.0,338.0,11.0
145,PINOT Thibaut,FDJ,115.0,0.0,104.0,17.0,58.0,0.0,22.0,326.0,10.0
167,SAGAN Peter,Tinkoff - Saxo,154.0,0.0,30.0,0.0,82.0,0.0,0.0,286.0,20.0
70,FROOME Chris,Team Sky,123.0,0.0,89.0,8.0,22.0,0.0,26.0,268.0,0.0
186,THOMAS Geraint,Team Sky,119.0,0.0,120.0,0.0,0.0,0.0,12.0,254.0,3.0


In [105]:
score_mapping = {
    "Stage": ["Stage"], 
    "GC": ["GC", 'Youth'], 
    "KOM": ['KOM','IntermediateKOM'], 
    'Sprint':['Sprint', 'IntermediateSprint'],
    'Total': ['Total']
}
combined_interval_score = combine_scores(interval_score, score_mapping)
combined_interval_score.sort_values(by='Total', ascending=False).head(10)

,Stage,GC,KOM,Sprint,Total,Rider,Team
62,134.0,109.0,12.0,113.0,338.0,DUMOULIN Tom,Team Giant - Alpecin
145,115.0,104.0,83.0,88.0,326.0,PINOT Thibaut,FDJ
167,154.0,30.0,0.0,142.0,286.0,SAGAN Peter,Tinkoff - Saxo
70,123.0,89.0,86.0,22.0,268.0,FROOME Chris,Team Sky
186,119.0,120.0,36.0,9.0,254.0,THOMAS Geraint,Team Sky
201,99.0,104.0,45.0,18.0,236.0,VAN GARDEREN Tejay,BMC Racing Team
218,100.0,104.0,18.0,25.0,221.0,ŠPILAK Simon,Team Katusha
9,84.0,114.0,63.0,12.0,199.0,BARDET Romain,AG2R La Mondiale
215,89.0,152.0,39.0,2.0,196.0,YATES Simon,Orica GreenEDGE
26,90.0,40.0,0.0,84.0,196.0,BOUHANNI Nacer,"Cofidis, Solutions Crédits"


In [110]:
intscore = combined_interval_score.set_index(keys=['Rider', 'Team'])
intscore.columns = intscore.columns.map(lambda x: x + "_" + str(0))
intscore = intscore.reset_index()
intscore.head(5)

,Rider,Team,Stage_0,GC_0,KOM_0,Sprint_0,Total_0
0,AGNOLI Valerio,Astana Pro Team,0.0,0.0,26.0,0.0,10.0
1,ALAPHILIPPE Julian,Etixx - Quick Step,42.0,31.0,0.0,1.0,54.0
2,ALBASINI Michael,Orica GreenEDGE,44.0,0.0,12.0,0.0,48.0
3,ANACONA Winner,Movistar Team,47.0,14.0,3.0,0.0,62.0
4,ANTÓN Igor,Movistar Team,0.0,0.0,0.0,0.0,2.0


In [64]:

score_template = get_score_template()


score_mapping = {}

data = []
for tdf in tdfs:
    tdf_scores = []
    intervals = [tdf.date - timedelta(30*i) for i in range(1,25)]
    for interval_idx, (interval_start, interval_end) in enumerate(pairwise(intervals)):
        interval_calendar = calendar_between(interval_start, interval_end)
        races = calendar_to_race_list(interval_calendar)
        interval_scores = [score_race(race, score_template) for race in races]
        if interval_scores == []:
            tdf_scores.append(None)
        else:
            scores = pd.concat(interval_scores, axis=0)
            interval_score = scores.groupby(by='Rider').agg('sum').reset_index()
            interval_score.columns = interval_score.columns.map(lambda x: x + "_" + interval_idx)
            tdf_scores.append(interval_score)
    
    tdf_data = pd.concat(tdf_scores, axis=1)

ValueError: All objects passed were None